# Classical Time Series Forecasting for Stock Prices

This notebook uses ARIMA/SARIMA, Prophet, and Holt-Winters (Exponential Smoothing) to model and forecast stock closing prices from `RELIANCE.csv`.

- Robust column inference for date and target (`Close`/`Adj Close`).
- Time-aware train/test split and evaluation (MAE, RMSE, MAPE).
- Future forecasting with the best model.

If any library is missing, use the setup cell below to install dependencies.


In [ ]:
# Optional: install core packages if missing
import sys, subprocess

def ensure(pkgs):
    to_install = []
    for p in pkgs:
        try:
            __import__(p)
        except Exception:
            to_install.append(p)
    if to_install:
        print("Installing:", to_install)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install])

# You can comment out packages you already have
ensure([
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit_learn",  # installed as scikit-learn
    "statsmodels", "pmdarima",
    "prophet"        # formerly fbprophet
])

# Now imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error

import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing

try:
    import pmdarima as pm
except Exception:
    pm = None

try:
    from prophet import Prophet
except Exception:
    Prophet = None

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
# Load data and infer columns
from pathlib import Path

file_path = Path("RELIANCE.csv")
assert file_path.exists(), f"CSV not found at {file_path.absolute()}"

df_raw = pd.read_csv(file_path)
print("Raw shape:", df_raw.shape)
print(df_raw.head())

# Infer date column
candidate_dates = [c for c in df_raw.columns if c.lower() in ["date","datetime","timestamp","time"]]
if not candidate_dates:
    # try first column if it looks like dates
    try:
        pd.to_datetime(df_raw.iloc[:,0])
        candidate_dates = [df_raw.columns[0]]
    except Exception:
        pass

if not candidate_dates:
    raise ValueError("Could not infer a date/datetime column. Please rename the date column to 'Date'.")


# Parse and sort by date
date_col = candidate_dates[0]
df = df_raw.copy()
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df = df.dropna(subset=[date_col]).sort_values(date_col)

# Infer target (Close/Adj Close)
candidate_targets = [c for c in df.columns if c.lower() in ["close","adj close","adj_close","price"]]
if not candidate_targets:
    # choose the first numeric column other than date
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not num_cols:
        raise ValueError("No numeric columns found for target variable.")
    candidate_targets = [num_cols[0]]

y_col = candidate_targets[0]
print(f"Using date column: {date_col} | target column: {y_col}")

# Build time series
series = df[[date_col, y_col]].dropna().copy()
series = series.groupby(date_col, as_index=False)[y_col].last()
series = series.set_index(date_col).asfreq('D')  # daily frequency; may create gaps
series[y_col] = series[y_col].interpolate(method='time')  # fill small gaps

series.head(), series.tail()


In [ ]:
# Quick EDA: summary and plots
print(series.describe())

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
series[y_col].plot(ax=ax[0], title=f"{y_col} over time")
series[y_col].pct_change().mul(100).plot(ax=ax[1], title="Daily return (%)")
plt.tight_layout()
plt.show()

try:
    from pandas.plotting import autocorrelation_plot
    fig, ax = plt.subplots(1,2, figsize=(12,4))
    sm.graphics.tsa.plot_acf(series[y_col].dropna(), lags=60, ax=ax[0])
    sm.graphics.tsa.plot_pacf(series[y_col].dropna(), lags=60, ax=ax[1], method='ywm')
    ax[0].set_title('ACF'); ax[1].set_title('PACF');
    plt.tight_layout(); plt.show()
except Exception as e:
    print("ACF/PACF plot skipped:", e)


In [ ]:
# Train/Test split (time-based)
from datetime import timedelta

# Parameters
test_size_days = 180   # adjust as needed
forecast_horizon = 30  # future days to forecast later

cutoff_date = series.index.max() - pd.Timedelta(days=test_size_days)
train = series.loc[:cutoff_date].copy()
_test = series.loc[cutoff_date+pd.Timedelta(days=1):].copy()

print("Train range:", train.index.min(), "to", train.index.max(), "| n=", len(train))
print("Test  range:", _test.index.min(), "to", _test.index.max(), "| n=", len(_test))

fig, ax = plt.subplots(figsize=(12,4))
train[y_col].plot(ax=ax, label='Train')
_test[y_col].plot(ax=ax, label='Test')
ax.legend(); ax.set_title('Train/Test split');
plt.show()


In [ ]:
# Helpers: metrics and container for results
from math import sqrt

def rmse(y_true, y_pred):
    return sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    yt = np.array(y_true)
    yp = np.array(y_pred)
    return np.mean(np.abs((yt - yp) / np.clip(np.abs(yt), 1e-8, None))) * 100.0

results = []  # list of dicts
fitted_models = {}  # name -> model objects and extras


In [ ]:
# ARIMA/SARIMA via pmdarima auto_arima (fallback to simple ARIMA if pmdarima missing)
name = "ARIMA/SARIMA"

try:
    if pm is not None:
        seasonal = True  # allow seasonal search; daily data may have weekly seasonality
        m = 7  # weekly period
        auto_model = pm.auto_arima(
            train[y_col],
            seasonal=seasonal,
            m=m,
            stepwise=True,
            suppress_warnings=True,
            error_action='ignore',
            trace=False
        )
        order = auto_model.order
        sorder = auto_model.seasonal_order if seasonal else (0,0,0,0)
        print("Selected order:", order, "seasonal:", sorder)

        sarimax = sm.tsa.statespace.SARIMAX(
            train[y_col],
            order=order,
            seasonal_order=sorder,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
    else:
        print("pmdarima not available; fitting basic ARIMA(1,1,1)")
        sarimax = sm.tsa.statespace.SARIMAX(train[y_col], order=(1,1,1), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

    pred = sarimax.get_prediction(start=_test.index[0], end=_test.index[-1])
    yhat = pred.predicted_mean.reindex(_test.index)

    mae = mean_absolute_error(_test[y_col], yhat)
    r = {
        "model": name,
        "MAE": mae,
        "RMSE": rmse(_test[y_col], yhat),
        "MAPE": mape(_test[y_col], yhat)
    }
    results.append(r)
    fitted_models[name] = {"model": sarimax}

    ax = train[y_col].plot(label='Train')
    _test[y_col].plot(ax=ax, label='Test')
    yhat.plot(ax=ax, label=f'{name} pred')
    ax.legend(); ax.set_title(f'{name} — test predictions')
    plt.show()
except Exception as e:
    print(f"{name} failed:", e)


In [ ]:
# Prophet model
name = "Prophet"

try:
    if Prophet is None:
        raise ImportError("prophet not installed")

    df_train = train.reset_index().rename(columns={train.index.name: 'ds', y_col: 'y'})
    df_test = _test.reset_index().rename(columns={_test.index.name: 'ds', y_col: 'y'})

    m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(df_train)

    forecast = m.predict(df_test[['ds']])
    yhat = forecast['yhat'].values

    mae = mean_absolute_error(df_test['y'], yhat)
    r = {
        "model": name,
        "MAE": mae,
        "RMSE": rmse(df_test['y'].values, yhat),
        "MAPE": mape(df_test['y'].values, yhat)
    }
    results.append(r)
    fitted_models[name] = {"model": m}

    ax = train[y_col].plot(label='Train')
    _test[y_col].plot(ax=ax, label='Test')
    pd.Series(yhat, index=_test.index, name='Prophet').plot(ax=ax, label='Prophet pred')
    ax.legend(); ax.set_title('Prophet — test predictions')
    plt.show()
except Exception as e:
    print(f"{name} failed:", e)


In [ ]:
# Holt-Winters (Exponential Smoothing)
name = "Holt-Winters"

try:
    seasonal_periods = 7  # weekly pattern; adjust if needed
    hw = ExponentialSmoothing(
        train[y_col],
        trend='add',
        seasonal='add',
        seasonal_periods=seasonal_periods,
        initialization_method='estimated'
    ).fit()

    yhat = hw.forecast(len(_test))
    yhat = pd.Series(yhat.values, index=_test.index)

    mae = mean_absolute_error(_test[y_col], yhat)
    r = {
        "model": name,
        "MAE": mae,
        "RMSE": rmse(_test[y_col], yhat),
        "MAPE": mape(_test[y_col], yhat)
    }
    results.append(r)
    fitted_models[name] = {"model": hw, "seasonal_periods": seasonal_periods}

    ax = train[y_col].plot(label='Train')
    _test[y_col].plot(ax=ax, label='Test')
    yhat.plot(ax=ax, label='Holt-Winters pred')
    ax.legend(); ax.set_title('Holt-Winters — test predictions')
    plt.show()
except Exception as e:
    print(f"{name} failed:", e)


In [ ]:
# Compare models
if results:
    results_df = pd.DataFrame(results).sort_values("RMSE")
    print(results_df)
else:
    results_df = pd.DataFrame()
    print("No models produced results.")


In [ ]:
# Future forecast using the best model
best_name = None
if 'results_df' in globals() and not results_df.empty:
    best_name = results_df.iloc[0]['model']
    print("Best model:", best_name)
else:
    print("No best model available.")

future_forecast = None
if best_name is not None:
    if best_name.startswith('ARIMA'):
        m = fitted_models[best_name]["model"]
        fc = m.get_forecast(steps=forecast_horizon)
        future_index = pd.date_range(series.index.max() + pd.Timedelta(days=1), periods=forecast_horizon, freq='D')
        future_forecast = pd.Series(fc.predicted_mean.values, index=future_index, name='forecast')
    elif best_name == 'Prophet':
        m = fitted_models[best_name]["model"]
        last_day = series.index.max()
        future_df = pd.DataFrame({"ds": pd.date_range(last_day + pd.Timedelta(days=1), periods=forecast_horizon, freq='D')})
        fc = m.predict(future_df)
        future_forecast = pd.Series(fc['yhat'].values, index=future_df['ds'], name='forecast')
    elif best_name == 'Holt-Winters':
        m = fitted_models[best_name]["model"]
        future_index = pd.date_range(series.index.max() + pd.Timedelta(days=1), periods=forecast_horizon, freq='D')
        fc = m.forecast(forecast_horizon)
        future_forecast = pd.Series(fc.values, index=future_index, name='forecast')

if future_forecast is not None:
    ax = series[y_col].plot(label='History')
    future_forecast.plot(ax=ax, label=f'{best_name} forecast', linestyle='--')
    ax.legend(); ax.set_title(f'Next {forecast_horizon} days forecast')
    plt.show()

    out = pd.DataFrame({
        'date': future_forecast.index,
        'forecast': future_forecast.values
    })
    out_path = Path('future_forecast.csv')
    out.to_csv(out_path, index=False)
    print(f"Saved future forecast to {out_path.resolve()}")
else:
    print("Future forecast not generated.")
